# V5 Notebook 01 — MLOps Foundations

**Production unit:** PU-B05-C01  
**Case:** a synthetic agricultural-risk classification service  
**Purpose:** create reproducible evidence from data acceptance through governed promotion and rollback.  

All data are synthetic. The notebook is independently runnable and writes evidence to `artifacts/pu_b05_c01`.

## Learning objectives

By the end, you can: define a reproducibility contract; validate data; build deterministic splits; compare a baseline and candidate; record lineage; apply acceptance gates; simulate a registry, drift monitoring, approval and rollback; and package auditable evidence.

In [1]:
from pathlib import Path
import json, hashlib, platform, sys, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
import joblib
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
ARTIFACT_DIR = Path('artifacts/pu_b05_c01')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'seed': SEED})

{'python': '3.12.14', 'platform': 'Linux-6.18.44-x86_64-with-glibc2.39', 'seed': 42}


## 1. Reproducibility contract

A reproducibility contract identifies the data, code, configuration, environment and random state needed to reconstruct a declared result. It does not promise bitwise equality across every platform; it states what is controlled and what tolerance is acceptable.

In [2]:
config = {
    'production_unit': 'PU-B05-C01', 'model_name': 'agricultural-risk-model',
    'candidate_version': '0.1.0', 'seed': SEED, 'test_size': 0.20,
    'validation_size_within_train': 0.25,
    'thresholds': {'roc_auc_min': 0.75, 'recall_min': 0.70, 'drift_psi_max': 0.20},
    'governance_approval': False
}
(ARTIFACT_DIR/'config.json').write_text(json.dumps(config, indent=2))
config

## 2. Generate a synthetic agricultural-risk dataset

The unit of analysis is a farm-season record. The target indicates elevated production risk. Region is included to support slice-based evaluation. No record represents a real farm or person.

In [3]:
n = 1800
rng = np.random.default_rng(SEED)
region = rng.choice(['North','Central','South','East'], n, p=[.24,.30,.28,.18])
rainfall = np.clip(rng.normal(620, 150, n), 180, 1100)
temperature = rng.normal(27.5, 2.8, n)
soil_index = np.clip(rng.beta(4, 2, n), 0, 1)
market_distance = rng.gamma(2.2, 18, n)
irrigated = rng.binomial(1, .34, n)
farm_size = np.clip(rng.lognormal(1.0, .65, n), .2, 35)
logit = (-1.1 - .006*(rainfall-600) + .30*(temperature-27) - 2.0*(soil_index-.5)
         + .012*(market_distance-35) - .70*irrigated + .10*(farm_size-2.5)
         + np.where(region=='North', .35, 0))
prob = 1/(1+np.exp(-logit))
risk = rng.binomial(1, prob)
df = pd.DataFrame({'region':region,'rainfall_mm':rainfall.round(1),
 'temperature_c':temperature.round(2),'soil_index':soil_index.round(3),
 'market_distance_km':market_distance.round(1),'irrigated':irrigated,
 'farm_size_ha':farm_size.round(2),'high_risk':risk})
df.to_csv(ARTIFACT_DIR/'synthetic_agricultural_risk.csv', index=False)
df.head()

## 3. Data acceptance checks

Acceptance tests are executable statements about schema, ranges, uniqueness and missingness. A failed mandatory check blocks training until the issue is remediated or an authorized exception is recorded.

In [4]:
expected = ['region','rainfall_mm','temperature_c','soil_index','market_distance_km','irrigated','farm_size_ha','high_risk']
checks = {
 'schema_exact': list(df.columns)==expected,
 'row_count_min_1000': len(df)>=1000,
 'target_binary': set(df.high_risk.unique()).issubset({0,1}),
 'rainfall_range': df.rainfall_mm.between(0,1500).all(),
 'soil_index_range': df.soil_index.between(0,1).all(),
 'missing_rate_below_0_02': df.isna().mean().max()<.02
}
assert all(checks.values()), checks
pd.Series(checks, name='pass').to_csv(ARTIFACT_DIR/'data_acceptance.csv')
checks

## 4. Fingerprint the accepted data

A digest identifies bytes, not meaning. It is useful only when paired with schema, provenance, time coverage and acceptance results.

In [5]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(65536),b''): h.update(block)
    return h.hexdigest()
data_path=ARTIFACT_DIR/'synthetic_agricultural_risk.csv'
data_digest=sha256(data_path)
data_digest

## 5. Deterministic train, validation and test splits

The test set remains outside model selection. Stratification preserves the target distribution. The split manifest records membership so later reruns do not silently change the information boundary.

In [6]:
X=df.drop(columns='high_risk'); y=df.high_risk
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.20,random_state=SEED,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,random_state=SEED,stratify=y_dev)
split_summary=pd.DataFrame({'rows':[len(X_train),len(X_val),len(X_test)],
 'positive_rate':[y_train.mean(),y_val.mean(),y_test.mean()]},index=['train','validation','test'])
split_summary.to_csv(ARTIFACT_DIR/'split_summary.csv')
split_summary

## 6. Establish a baseline

The prevalence baseline predicts the development-set majority class. A candidate that cannot beat a declared baseline should not advance merely because it uses a sophisticated algorithm.

In [7]:
majority=int(y_train.mean()>=.5)
baseline_pred=np.repeat(majority,len(y_val))
baseline={'accuracy':accuracy_score(y_val,baseline_pred),'recall':recall_score(y_val,baseline_pred,zero_division=0)}
baseline

## 7. Build a portable preprocessing and model pipeline

The pipeline binds transformations to the estimator, reducing training-serving skew. Unknown categories are handled explicitly.

In [8]:
numeric=['rainfall_mm','temperature_c','soil_index','market_distance_km','irrigated','farm_size_ha']
categorical=['region']
prep=ColumnTransformer([
 ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),
 ('cat',OneHotEncoder(handle_unknown='ignore'),categorical)])
model=Pipeline([('preprocess',prep),('classifier',LogisticRegression(max_iter=1000,random_state=SEED))])
model.fit(X_train,y_train)
model

## 8. Evaluate the candidate on validation data

Threshold-dependent metrics answer different operational questions. Recall matters when failing to flag a genuinely high-risk case is costly; precision matters when interventions are scarce.

In [9]:
def metrics(y_true,p,threshold=.5):
    pred=(p>=threshold).astype(int)
    return {'accuracy':accuracy_score(y_true,pred),'precision':precision_score(y_true,pred),
            'recall':recall_score(y_true,pred),'f1':f1_score(y_true,pred),'roc_auc':roc_auc_score(y_true,p)}
val_prob=model.predict_proba(X_val)[:,1]
val_metrics=metrics(y_val,val_prob)
pd.Series(val_metrics).round(4)

## 9. Inspect errors and regional slices

Aggregate performance can hide weak service for a region. Slice evidence informs investigation; it does not by itself establish causality or fairness.

In [10]:
val_eval=X_val[['region']].copy(); val_eval['y']=y_val.values; val_eval['p']=val_prob
val_eval['pred']=(val_eval.p>=.5).astype(int)
slice_rows=[]
for region_name,g in val_eval.groupby('region'):
    slice_rows.append({'region':region_name,'n':len(g),'recall':recall_score(g.y,g.pred),
                       'precision':precision_score(g.y,g.pred),'positive_rate':g.y.mean()})
slice_metrics=pd.DataFrame(slice_rows)
slice_metrics.to_csv(ARTIFACT_DIR/'regional_slice_metrics.csv',index=False)
slice_metrics

In [11]:
fig,ax=plt.subplots(figsize=(7,4))
slice_metrics.set_index('region')[['recall','precision']].plot.bar(ax=ax,color=['#17365D','#7F8FA6'])
ax.set_ylim(0,1); ax.set_ylabel('Score'); ax.set_title('Validation performance by region')
ax.grid(axis='y',alpha=.25); fig.tight_layout(); fig.savefig(ARTIFACT_DIR/'regional_metrics.png',dpi=160); plt.show()

## 10. Record the experiment

An experiment record binds configuration, data identity, code intent and metrics. A folder is not a registry, but the transparent JSON record demonstrates the minimum evidence contract without external services.

In [12]:
run_record={'run_id':'pu-b05-c01-run-001','data_sha256':data_digest,'seed':SEED,
 'algorithm':'LogisticRegression','validation_metrics':val_metrics,'parameters':{'max_iter':1000}}
(ARTIFACT_DIR/'run_record.json').write_text(json.dumps(run_record,indent=2))
run_record

## 11. Serialize and fingerprint the candidate

The model artifact is saved only after its inputs and evaluation record exist. Its digest becomes part of the release manifest.

In [13]:
model_path=ARTIFACT_DIR/'agricultural_risk_model.joblib'
joblib.dump(model,model_path)
model_digest=sha256(model_path)
model_digest

## 12. Test an inference contract

The contract declares required fields and output semantics. Tests cover valid input, missing fields and unseen categories before any deployment claim.

In [14]:
sample=X_test.iloc[[0]].copy()
required=set(X.columns)
def predict_contract(frame):
    missing=required-set(frame.columns)
    if missing: raise ValueError(f'Missing fields: {sorted(missing)}')
    p=float(model.predict_proba(frame[list(X.columns)])[:,1][0])
    return {'model_version':'0.1.0','risk_probability':round(p,6),'high_risk':bool(p>=.5)}
valid_result=predict_contract(sample)
try:
    predict_contract(sample.drop(columns='rainfall_mm'))
    missing_field_test=False
except ValueError:
    missing_field_test=True
{'valid':valid_result,'missing_field_rejected':missing_field_test}

## 13. Apply technical promotion gates

Gates are conjunctive. Averaging metrics can conceal a mandatory failure. Passing technical gates means eligible for review, not authorized for production.

In [15]:
technical_gates={
 'data_accepted':all(checks.values()),
 'roc_auc':val_metrics['roc_auc']>=config['thresholds']['roc_auc_min'],
 'recall':val_metrics['recall']>=config['thresholds']['recall_min'],
 'contract_test':missing_field_test,
 'artifact_saved':model_path.exists()
}
technical_ready=all(technical_gates.values())
{'gates':technical_gates,'technical_ready':technical_ready}

## 14. Final test evaluation

The test set is used once after the candidate and threshold are fixed. This result estimates generalization under the declared information boundary; it does not guarantee future operational performance.

In [16]:
test_prob=model.predict_proba(X_test)[:,1]
test_metrics=metrics(y_test,test_prob)
pd.Series(test_metrics).round(4).to_csv(ARTIFACT_DIR/'test_metrics.csv')
pd.Series(test_metrics).round(4)

## 15. Simulate production drift

Population Stability Index (PSI) summarizes distributional change across bins. It is a monitoring signal, not proof of model failure or causality.

In [17]:
def psi(expected,actual,bins=10):
    edges=np.quantile(expected,np.linspace(0,1,bins+1)); edges[0],edges[-1]=-np.inf,np.inf
    e=np.histogram(expected,bins=edges)[0]/len(expected)
    a=np.histogram(actual,bins=edges)[0]/len(actual)
    e=np.clip(e,1e-6,None); a=np.clip(a,1e-6,None)
    return float(np.sum((a-e)*np.log(a/e)))
production_rainfall=np.clip(rng.normal(520,170,500),120,1100)
rainfall_psi=psi(X_train.rainfall_mm.values,production_rainfall)
rainfall_psi

In [18]:
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(X_train.rainfall_mm,bins=20,alpha=.65,label='Training',color='#17365D',density=True)
ax.hist(production_rainfall,bins=20,alpha=.55,label='Production window',color='#C55A11',density=True)
ax.set(title=f'Rainfall monitoring window — PSI {rainfall_psi:.3f}',xlabel='Rainfall mm',ylabel='Density')
ax.legend(); fig.tight_layout(); fig.savefig(ARTIFACT_DIR/'rainfall_drift.png',dpi=160); plt.show()

## 16. Governance approval and fail-closed release

Automation assembles evidence; an accountable authority decides whether the declared use is acceptable. The canonical example keeps approval false so the initial release is correctly blocked.

In [19]:
monitoring_ready=rainfall_psi<=config['thresholds']['drift_psi_max']
release_gates={**technical_gates,'monitoring_plan':monitoring_ready,
               'rollback_tested':True,'governance_approval':config['governance_approval']}
authorized_release=all(release_gates.values())
assert authorized_release is False
{'release_gates':release_gates,'authorized_release':authorized_release,
 'decision':'BLOCKED — obtain named governance approval and address any failed monitoring gate'}

## 17. Create the release manifest and model card

The manifest identifies exact artifacts. The model card documents intended use, limitations, evaluation and ownership. Neither document substitutes for independent review or authorization.

In [20]:
manifest={'production_unit':'PU-B05-C01','candidate':'0.1.0','data_sha256':data_digest,
 'model_sha256':model_digest,'test_metrics':test_metrics,'release_gates':release_gates,
 'authorized_release':authorized_release}
def json_ready(value):
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, dict): return {str(k):json_ready(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)): return [json_ready(v) for v in value]
    return value
manifest=json_ready(manifest)
(ARTIFACT_DIR/'release_manifest.json').write_text(json.dumps(manifest,indent=2))
model_card=f'''# Model Card — Synthetic Agricultural Risk Model

Intended use: education and controlled workflow demonstration only.
Not intended for real farm, credit, insurance or benefit decisions.
Data: synthetic. Seed: {SEED}.
Test ROC AUC: {test_metrics['roc_auc']:.3f}. Test recall: {test_metrics['recall']:.3f}.
Limitations: simulated relationships; no causal validity; no external validation.
Release decision: BLOCKED pending governance approval.
'''
(ARTIFACT_DIR/'MODEL_CARD.md').write_text(model_card)
print(model_card)

# Model Card — Synthetic Agricultural Risk Model

Intended use: education and controlled workflow demonstration only.
Not intended for real farm, credit, insurance or benefit decisions.
Data: synthetic. Seed: 42.
Test ROC AUC: 0.757. Test recall: 0.326.
Limitations: simulated relationships; no causal validity; no external validation.
Release decision: BLOCKED pending governance approval.



## 18. Simulate rollback

Rollback restores a complete known-good manifest, not merely an older model file. The record below demonstrates the decision path and the required evidence.

In [21]:
rollback_record={'trigger':'post-deployment acceptance failure','from_version':'0.1.0',
 'to_version':'0.0.1','data_contract_restored':True,'service_contract_restored':True,
 'owner':'platform-operations','incident_record_required':True,'status':'SIMULATED_PASS'}
(ARTIFACT_DIR/'rollback_record.json').write_text(json.dumps(rollback_record,indent=2))
rollback_record

## 19. Evidence inventory and package

The final inventory makes missing evidence visible. Re-running this notebook refreshes the evidence rather than silently relying on stale files.

In [22]:
inventory=[]
for p in sorted(ARTIFACT_DIR.iterdir()):
    if p.is_file(): inventory.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256(p)})
pd.DataFrame(inventory).to_csv(ARTIFACT_DIR/'evidence_inventory.csv',index=False)
zip_path=Path('PU-B05-C01_Notebook_Evidence.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ARTIFACT_DIR.iterdir()):
        if p.is_file(): z.write(p,p.relative_to(ARTIFACT_DIR.parent))
pd.DataFrame(inventory)

## Student exercises

1. Raise the recall threshold to 0.80 and explain the operational trade-off.  
2. Introduce a schema failure and confirm that training is blocked.  
3. Compare PSI for rainfall and temperature.  
4. Add a regional minimum-recall gate.  
5. Change governance approval only after writing an approval record; explain why a Boolean alone is insufficient.  
6. Design a retirement gate for a model with no remaining authorized use.

## Interpretation and limitations

This notebook demonstrates evidence flow, not a production platform. Real deployment requires authenticated services, protected secrets, independent validation, environment-specific testing, access control, monitoring infrastructure, incident response and legally authorized governance.